# Feature Extraction

This is where raw text becomes numbers the models can actually learn from.

We compute five feature groups per response: token length, lexical diversity (type-token ratio), readability (Flesch score), overlap with the reference (ROUGE-L), and semantic similarity (sentence embeddings). Everything gets cached to parquet so we never recompute unless we deliberately ask to.

A word on what we're featurizing: HH-RLHF stores full conversations, but we extracted just the final assistant response in the loader. That's what we're scoring here, which is what actually matters.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd

from inference_lens.features.extractor import (
    compute_lexical_features,
    compute_embeddings,
    compute_rouge_l,
    extract_all_features,
)
from inference_lens.utils.logging import setup_logging

setup_logging()
pd.set_option('display.max_colwidth', 100)

## Load the flattened dataset

Picking up from where notebook 01 left off.

In [ ]:
df = pd.read_parquet('../../data/processed/hh_rlhf_flat.parquet')
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
df.head(2)

## Sanity check on response extraction

Before computing features on 170K rows, let's verify the assistant response extraction actually worked on a few examples.

In [ ]:
sample = df.sample(3, random_state=0)

for i, row in sample.iterrows():
    print(f'--- row {i} ---')
    print(f'chosen_response  : {row["chosen_response"][:200]}')
    print(f'rejected_response: {row["rejected_response"][:200]}')
    print()

## Dry run on 500 rows

Always run on a small slice before committing to the full dataset. Catches bugs cheaply.

In [ ]:
sample_df = df.sample(500, random_state=42).copy()

# test lexical features first since they're fast
chosen_texts = sample_df['chosen_response'].tolist()
rejected_texts = sample_df['rejected_response'].tolist()

lex = compute_lexical_features(chosen_texts)
print('Lexical features shape:', lex.shape)
lex.describe().round(2)

In [ ]:
# ROUGE-L on the sample
# reference is chosen_response itself since we're measuring how much the chosen
# response overlaps with the rejected one -- informative for understanding divergence
rouge_scores = compute_rouge_l(rejected_texts, chosen_texts)
print(f'ROUGE-L (rejected vs chosen) -- mean: {np.mean(rouge_scores):.3f}  median: {np.median(rouge_scores):.3f}')

In [ ]:
# embeddings on the sample -- takes a minute or so
emb = compute_embeddings(chosen_texts[:100])
print(f'Embeddings shape: {emb.shape}  dtype: {emb.dtype}')
print(f'Norm check (should be ~1.0): {np.linalg.norm(emb[0]):.4f}')

## Full feature extraction

Now we run on the complete dataset. This will take a while on first run (embeddings are the slow part). Subsequent runs skip straight to loading from cache.

We run feature extraction separately on chosen and rejected responses, then concatenate with a label column.

In [ ]:
import os
os.makedirs('../../data/processed', exist_ok=True)

# build a long-form dataframe: each row is one response with a label
# chosen = 1, rejected = 0
chosen_df = df[['chosen_response', 'subset', 'original_split']].copy()
chosen_df = chosen_df.rename(columns={'chosen_response': 'response'})
chosen_df['label'] = 1

rejected_df = df[['rejected_response', 'subset', 'original_split']].copy()
rejected_df = rejected_df.rename(columns={'rejected_response': 'response'})
rejected_df['label'] = 0

long_df = pd.concat([chosen_df, rejected_df], ignore_index=True)
long_df = long_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'Long-form dataset: {len(long_df):,} rows')
print(f'Label balance: {long_df["label"].value_counts().to_dict()}')

In [ ]:
# save long-form before features so we have a clean checkpoint
long_df.to_parquet('../../data/processed/hh_rlhf_long.parquet', index=False)
print('Saved hh_rlhf_long.parquet')

In [ ]:
# compute all features and cache
# pass force_recompute=True if you change the feature set and need a fresh cache
features_df, embeddings = extract_all_features(
    long_df,
    text_col='response',
    reference_col='response',
    include_bertscore=False,   # too slow without GPU, skip for now
    features_cache='../../data/processed/features.parquet',
    embeddings_cache='../../data/processed/embeddings.npy',
    force_recompute=False,
)

print(f'Features shape: {features_df.shape}')
print(f'Embeddings shape: {embeddings.shape}')
features_df.describe().round(3)

## Attach labels and save final feature table

In [ ]:
features_df['label'] = long_df['label'].values
features_df['subset'] = long_df['subset'].values
features_df['original_split'] = long_df['original_split'].values

features_df.to_parquet('../../data/processed/features_with_labels.parquet', index=False)
print(f'Saved features_with_labels.parquet -- {len(features_df):,} rows, {features_df.shape[1]} columns')
features_df.head(3)

## Quick check: do features differ between chosen and rejected?

If our features have zero signal, the models have nothing to learn from. Let's see if the distributions look meaningfully different before moving to EDA.

In [ ]:
feature_cols = ['token_length', 'type_token_ratio', 'flesch_score', 'rouge_l']

print(f'{"Feature":<22} {"Chosen mean":>14} {"Rejected mean":>14} {"Diff":>10}')
print('-' * 64)
for col in feature_cols:
    chosen_mean = features_df[features_df['label'] == 1][col].mean()
    rejected_mean = features_df[features_df['label'] == 0][col].mean()
    diff = chosen_mean - rejected_mean
    print(f'{col:<22} {chosen_mean:>14.3f} {rejected_mean:>14.3f} {diff:>+10.3f}')